### Ingest races.csv file
1. Read the file using Spark dataframe reader API
2. Add metadata columns
    - Source File
    - Ingestion Timestamp
3. Write toi bronze delta table

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id = dbutils.widgets.get("p_batch_id")


In [0]:
%run ../00-Common/01.environment-config

In [0]:
%run ../00-Common/02.bronze-helpers

In [0]:
source_file = f"{landing_folder_path}/{v_batch_id}/races.csv"
table_name = f"{catalog_name}.{bronze_schema}.races"

In [0]:
from pyspark.sql import functions as f

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType
races_schema = StructType([
        StructField('season', IntegerType(), True),
        StructField('round', IntegerType(), True),
        StructField('url', StringType(), True),
        StructField('raceName', StringType(), True),
        StructField('date', DateType(), True),
        StructField('circuitId', StringType(), True)
    ])

In [0]:
races_df = (
    spark.read
        .format('csv')
        .schema(races_schema)
        .option('header', True)
        .option('mode','FAILFAST')
        .load('/Volumes/formula1/landing/files/races.csv')
)

In [0]:
races_final_df = (
    races_df
        .withColumn('current_timestamp', f.current_timestamp())
        .withColumn('source_file', f.col('_metadata.file_path'))
)

In [0]:
display(races_final_df)

In [0]:
#(
#    races_final_df
#        .write
#        .format('delta')
#        .mode('overwrite')
#        .saveAsTable(table_name)
#)

In [0]:
write_to_bronze(
    races_final_df,
    table_name,
    v_batch_id
)

In [0]:
#display(spark.table(table_name))

In [0]:
#spark.table('formula1_incr.bronze.races').display()